# Bölüm 5 — BÖLÜM 5: MAKİNE ÖĞRENMESİNE GİRİŞ VE REGRESYON ANALİZİ

**VERİ MADENCİLİĞİ VE MAKİNE ÖĞRENMESİ**  
*Python ile Temel Analitikten Büyük Veri ve Gerçek Zamanlı Sistemlere*

Bu defter, kitabın 5. bölümündeki tüm kod örneklerini içerir. Her hücrenin başlığı kitaptaki alt bölüme karşılık gelir.


In [ ]:
# Bu bölüm için gerekli paketler
!pip install -q matplotlib numpy pandas scikit-learn scipy statsmodels


## 5.1. Makine Öğrenmesi Paradigması


### D. Pekiştirmeli Öğrenme (Reinforcement Learning): Ödül Maksimizasyonu

`bolum05/05_01_01_d-pekistirmeli-ogrenme-odul-maksimizasyonu.py`

_Kitap: Kod 5.1_


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, silhouette_score

np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
# A. GÖZETİMLİ ÖĞRENME — Binary Sınıflandırma
# ════════════════════════════════════════════════════════════════════════════

# Sentetik veri: 2 sınıf, 2 öznitelik
X_sup, y_sup = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                    n_informative=2, n_clusters_per_class=1,
                                    class_sep=1.5, random_state=42)

# Train/Test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X_sup, y_sup, test_size=0.3, random_state=42)

# Model: Lojistik Regresyon
# Amaç: f: ℝ² → {0,1} fonksiyonunu öğren
model_sup = LogisticRegression()
model_sup.fit(X_train, y_train)

# Tahmin
y_pred = model_sup.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("═══ GÖZETİMLİ ÖĞRENME ═══")
print(f"Eğitim seti boyutu: {len(X_train)}")
print(f"Test seti boyutu  : {len(X_test)}")
print(f"Model Accuracy    : {accuracy:.3f}")
print(f"Öğrenilen ağırlıklar: {model_sup.coef_[0]}")
print(f"Kesişim (intercept) : {model_sup.intercept_[0]:.3f}")

# Görselleştirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Eğitim verisi + karar sınırı
ax1.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1],
            c='blue', label='Sınıf 0 (Eğitim)', alpha=0.6, s=40)
ax1.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1],
            c='red', label='Sınıf 1 (Eğitim)', alpha=0.6, s=40)

# Karar sınırı: w₁x₁ + w₂x₂ + b = 0  →  x₂ = -(w₁x₁ + b)/w₂
w1, w2 = model_sup.coef_[0]
b = model_sup.intercept_[0]
x1_line = np.linspace(X_train[:, 0].min(), X_train[:, 0].max(), 100)
x2_line = -(w1*x1_line + b) / w2
ax1.plot(x1_line, x2_line, 'k--', lw=2, label='Karar Sınırı')
ax1.set_title('Gözetimli: Eğitim + Karar Sınırı', fontweight='bold')
ax1.set_xlabel('Öznitelik 1'); ax1.set_ylabel('Öznitelik 2')
ax1.legend(); ax1.grid(alpha=0.3)

# Sağ: Test tahminleri
correct = (y_pred == y_test)
ax2.scatter(X_test[correct, 0], X_test[correct, 1],
            c='green', marker='o', label='Doğru Tahmin', alpha=0.7, s=60)
ax2.scatter(X_test[~correct, 0], X_test[~correct, 1],
            c='orange', marker='X', label='Yanlış Tahmin', s=100)
ax2.plot(x1_line, x2_line, 'k--', lw=2, label='Karar Sınırı')
ax2.set_title(f'Test Performansı (Acc: {accuracy:.2%})', fontweight='bold')
ax2.set_xlabel('Öznitelik 1'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/supervised_demo.png', dpi=120, bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# B. GÖZETİMSİZ ÖĞRENME — Kümeleme (K-Means)
# ════════════════════════════════════════════════════════════════════════════

# Sentetik veri: 4 doğal küme, ETİKETSİZ
X_unsup, y_true = make_blobs(n_samples=400, centers=4, n_features=2,
                              cluster_std=1.2, random_state=42)

# Gözetimsiz model: K-Means
# Amaç: X verilerini K kümeye böl, öyle ki küme içi varyans minimal olsun
k = 4
model_unsup = KMeans(n_clusters=k, random_state=42, n_init=10)
y_pred_unsup = model_unsup.fit_predict(X_unsup)

# Silhouette Score: Kümeleme kalitesi (-1 kötü, +1 mükemmel)
silhouette = silhouette_score(X_unsup, y_pred_unsup)

print("\n═══ GÖZETİMSİZ ÖĞRENME ═══")
print(f"Veri boyutu       : {len(X_unsup)}")
print(f"Küme sayısı (K)   : {k}")
print(f"Silhouette Score  : {silhouette:.3f}")
print(f"Küme merkezleri:\n{model_unsup.cluster_centers_}")

# Görselleştirme
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Gerçek (gizli) yapı
for i in range(4):
    ax1.scatter(X_unsup[y_true==i, 0], X_unsup[y_true==i, 1],
                label=f'Gerçek Küme {i}', alpha=0.6, s=40)
ax1.set_title('Gerçek (Gizli) Küme Yapısı', fontweight='bold')
ax1.set_xlabel('Öznitelik 1'); ax1.set_ylabel('Öznitelik 2')
ax1.legend(); ax1.grid(alpha=0.3)

# Sağ: K-Means tarafından bulunan kümeler
colors = ['red', 'blue', 'green', 'purple']
for i in range(k):
    ax2.scatter(X_unsup[y_pred_unsup==i, 0], X_unsup[y_pred_unsup==i, 1],
                c=colors[i], label=f'Bulunan Küme {i}', alpha=0.6, s=40)
# Küme merkezlerini işaretle
ax2.scatter(model_unsup.cluster_centers_[:, 0],
            model_unsup.cluster_centers_[:, 1],
            c='black', marker='X', s=200, label='Merkezler',
            edgecolors='yellow', linewidths=2)
ax2.set_title(f'K-Means Kümeleme (Silhouette: {silhouette:.3f})', fontweight='bold')
ax2.set_xlabel('Öznitelik 1'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/unsupervised_demo.png', dpi=120, bbox_inches='tight')
plt.close()

print("\nGrafikler kaydedildi: /tmp/supervised_demo.png, /tmp/unsupervised_demo.png")


### C. Performans Metrikleri: Problem Tipine Göre Seçim

`bolum05/05_01_02_c-performans-metrikleri-problem-tipine-gore-seci.py`

_Kitap: Kod 5.2_


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression, load_iris
from sklearn.model_selection import (KFold, cross_val_score, cross_validate,
                                     learning_curve, validation_curve)
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, f1_score
import pandas as pd

np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
# A. REGRESYON: K-FOLD CV İLE MODEL DEĞERLENDİRME
# ════════════════════════════════════════════════════════════════════════════

# Sentetik regresyon verisi
X_reg, y_reg = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)

# Model: Ridge Regresyon (alpha=1.0)
model_ridge = Ridge(alpha=1.0)

# K-Fold tanımı (K=5, karıştırmalı)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Çapraz doğrulama skorları (negatif MSE)
cv_scores = cross_val_score(model_ridge, X_reg, y_reg, cv=kfold,
                             scoring='neg_mean_squared_error')

# Mutlak değere çevir (MSE pozitif olmalı)
mse_scores = -cv_scores
rmse_scores = np.sqrt(mse_scores)

print("═══ REGRESYON: 5-FOLD CROSS-VALIDATION ═══")
print(f"Fold MSE Skorları : {mse_scores}")
print(f"Ortalama MSE      : {mse_scores.mean():.2f} ± {mse_scores.std():.2f}")
print(f"Ortalama RMSE     : {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f}")

# R² skorları da hesaplayalım
cv_r2 = cross_val_score(model_ridge, X_reg, y_reg, cv=kfold, scoring='r2')
print(f"\nFold R² Skorları  : {cv_r2}")
print(f"Ortalama R²       : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

# Görselleştirme: Fold skorları
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

folds = np.arange(1, 6)
ax1.plot(folds, mse_scores, 'o-', color='#1E3A5F', markersize=10, lw=2, label='MSE')
ax1.axhline(mse_scores.mean(), color='red', ls='--', lw=2, label=f'Ortalama: {mse_scores.mean():.2f}')
ax1.fill_between(folds, mse_scores.mean()-mse_scores.std(),
                  mse_scores.mean()+mse_scores.std(),
                  alpha=0.2, color='red', label='±1 std')
ax1.set_xlabel('Fold #', fontsize=12); ax1.set_ylabel('MSE', fontsize=12)
ax1.set_title('K-Fold CV: MSE Skorları', fontweight='bold')
ax1.set_xticks(folds); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(folds, cv_r2, 's-', color='#2E8B57', markersize=10, lw=2, label='R²')
ax2.axhline(cv_r2.mean(), color='red', ls='--', lw=2, label=f'Ortalama: {cv_r2.mean():.3f}')
ax2.set_xlabel('Fold #', fontsize=12); ax2.set_ylabel('R²', fontsize=12)
ax2.set_title('K-Fold CV: R² Skorları', fontweight='bold')
ax2.set_xticks(folds); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/kfold_regression.png', dpi=120, bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# B. SINIFLANDIRMA: DETAYLI CV SONUÇLARI (cross_validate)
# ════════════════════════════════════════════════════════════════════════════

# Iris veri seti (çok sınıflı)
iris = load_iris()
X_clf, y_clf = iris.data, iris.target

# Model: Random Forest
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Çoklu metrik ile CV
scoring_metrics = {
    'accuracy': 'accuracy',
    'precision_macro': 'precision_macro',
    'recall_macro': 'recall_macro',
    'f1_macro': 'f1_macro'
}

cv_results = cross_validate(model_rf, X_clf, y_clf, cv=5,
                             scoring=scoring_metrics, return_train_score=True)

print("\n═══ SINIFLANDIRMA: 5-FOLD CV (ÇOKLU METRİK) ═══")
results_df = pd.DataFrame({
    'Fold': range(1, 6),
    'Train Acc': cv_results['train_accuracy'],
    'Test Acc': cv_results['test_accuracy'],
    'Test F1': cv_results['test_f1_macro'],
    'Test Precision': cv_results['test_precision_macro'],
    'Test Recall': cv_results['test_recall_macro']
})
print(results_df.to_string(index=False))
print(f"\nOrtalama Test Accuracy: {cv_results['test_accuracy'].mean():.3f}")
print(f"Ortalama Test F1-Score: {cv_results['test_f1_macro'].mean():.3f}")

# Train vs Test performans karşılaştırması (overfitting kontrolü)
fig, ax = plt.subplots(figsize=(10, 6))
folds = np.arange(1, 6)
width = 0.35
ax.bar(folds - width/2, cv_results['train_accuracy'], width,
       label='Train Accuracy', color='#1E3A5F', alpha=0.8)
ax.bar(folds + width/2, cv_results['test_accuracy'], width,
       label='Test Accuracy', color='#C44D34', alpha=0.8)
ax.axhline(cv_results['train_accuracy'].mean(), color='blue', ls='--', lw=1.5, alpha=0.7)
ax.axhline(cv_results['test_accuracy'].mean(), color='red', ls='--', lw=1.5, alpha=0.7)
ax.set_xlabel('Fold #', fontsize=12); ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Train vs Test Accuracy (Overfitting Check)', fontweight='bold')
ax.set_xticks(folds); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.savefig('/tmp/kfold_classification.png', dpi=120, bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# C. LEARNING CURVE: Eğitim Seti Boyutu vs Performans
# ════════════════════════════════════════════════════════════════════════════

# Learning curve: Farklı eğitim boyutlarında model performansı
train_sizes, train_scores, test_scores = learning_curve(
    model_rf, X_clf, y_clf, cv=5,
    train_sizes=np.linspace(0.1, 1.0, 10),
    scoring='accuracy', n_jobs=-1, random_state=42
)

# Ortalama ve std hesapla
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

print("\n═══ LEARNING CURVE ANALİZİ ═══")
print(f"Min eğitim boyutu: {train_sizes.min():.0f} örnek")
print(f"Max eğitim boyutu: {train_sizes.max():.0f} örnek")
print(f"Final test acc   : {test_mean[-1]:.3f} ± {test_std[-1]:.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_sizes, train_mean, 'o-', color='blue', lw=2, label='Train Score')
ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std,
                 alpha=0.2, color='blue')
ax.plot(train_sizes, test_mean, 's-', color='red', lw=2, label='CV Score (Test)')
ax.fill_between(train_sizes, test_mean-test_std, test_mean+test_std,
                 alpha=0.2, color='red')
ax.set_xlabel('Eğitim Seti Boyutu', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Learning Curve: Random Forest (Iris)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/learning_curve.png', dpi=120, bbox_inches='tight')
plt.close()

print("\nTüm grafikler kaydedildi: /tmp/kfold_*.png, /tmp/learning_curve.png")


### D. Görsel Analiz: Polinom Regresyon Örneği

`bolum05/05_01_03_d-gorsel-analiz-polinom-regresyon-ornegi.py`

_Kitap: Kod 5.3_


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

np.random.seed(42)

# ════════════════════════════════════════════════════════════════════════════
# A. SENTETİK VERİ: Gerçek fonksiyon + gürültü
# ════════════════════════════════════════════════════════════════════════════

# Gerçek fonksiyon: f(x) = x·sin(x)
n_samples = 50
X = np.sort(np.random.uniform(0, 10, n_samples))
y_true = X * np.sin(X)
noise = np.random.normal(0, 1.5, n_samples)
y = y_true + noise

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# Görselleştirme için yoğun grid
X_plot = np.linspace(0, 10, 500)
y_plot_true = X_plot * np.sin(X_plot)

# ════════════════════════════════════════════════════════════════════════════
# B. FARKLI POLİNOM DERECELERİ: d = 1, 3, 5, 15
# ════════════════════════════════════════════════════════════════════════════

degrees = [1, 3, 5, 15]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

train_errors = []
test_errors = []

for idx, degree in enumerate(degrees):
    ax = axes[idx]

    # Polinom öznitelikler
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly.fit_transform(X_train.reshape(-1, 1))
    X_test_poly = poly.transform(X_test.reshape(-1, 1))
    X_plot_poly = poly.transform(X_plot.reshape(-1, 1))

    # Model eğitimi
    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    # Tahminler
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    y_plot_pred = model.predict(X_plot_poly)

    # Hatalar
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_errors.append(train_mse)
    test_errors.append(test_mse)

    # Görselleştirme
    ax.scatter(X_train, y_train, color='blue', s=40, alpha=0.6, label='Train')
    ax.scatter(X_test, y_test, color='red', s=40, alpha=0.6, label='Test')
    ax.plot(X_plot, y_plot_true, 'k--', lw=1.5, alpha=0.5, label='Gerçek Fonksiyon')
    ax.plot(X_plot, y_plot_pred, 'g-', lw=2.5, label=f'Polinom (d={degree})')

    # Durum etiketi
    if degree == 1:
        durum = "UNDERFITTING"
        renk = 'orange'
    elif degree in [3, 5]:
        durum = "İYİ FIT"
        renk = 'green'
    else:
        durum = "OVERFITTING"
        renk = 'red'

    ax.text(0.5, 0.95, durum, transform=ax.transAxes, fontsize=12,
            fontweight='bold', color=renk, ha='center', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_title(f'Derece={degree}  |  Train MSE={train_mse:.2f}, Test MSE={test_mse:.2f}',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(loc='lower left', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(-12, 12)

plt.tight_layout()
plt.savefig('/tmp/bias_variance_polynomial.png', dpi=120, bbox_inches='tight')
plt.close()

# ════════════════════════════════════════════════════════════════════════════
# C. BIAS-VARIANCE AYRIŞIMI GRAFİĞİ
# ════════════════════════════════════════════════════════════════════════════

# Daha geniş derece aralığı
degrees_full = range(1, 21)
train_errors_full = []
test_errors_full = []

for d in degrees_full:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_tr_p = poly.fit_transform(X_train.reshape(-1, 1))
    X_te_p = poly.transform(X_test.reshape(-1, 1))

    model = LinearRegression()
    model.fit(X_tr_p, y_train)

    train_errors_full.append(mean_squared_error(y_train, model.predict(X_tr_p)))
    test_errors_full.append(mean_squared_error(y_test, model.predict(X_te_p)))

# Bias-Variance teorik eğrileri (illustrative)
bias_squared = np.linspace(20, 0.5, 20)   # Bias azalır
variance = np.linspace(0.5, 25, 20)        # Variance artar
total_error_theory = bias_squared + variance

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Sol: Gerçek Train/Test hataları
ax1.plot(degrees_full, train_errors_full, 'bo-', lw=2, label='Train MSE', markersize=6)
ax1.plot(degrees_full, test_errors_full, 'rs-', lw=2, label='Test MSE', markersize=6)
optimal_idx = np.argmin(test_errors_full)
ax1.axvline(degrees_full[optimal_idx], color='green', ls='--', lw=2,
            label=f'Optimal d={degrees_full[optimal_idx]}')
ax1.set_xlabel('Polinom Derecesi (Model Karmaşıklığı)', fontsize=12)
ax1.set_ylabel('MSE', fontsize=12)
ax1.set_title('Model Karmaşıklığı vs Hata', fontweight='bold')
ax1.legend(); ax1.grid(alpha=0.3)

# Sağ: Teorik Bias-Variance ayrışımı
ax2.plot(degrees_full, bias_squared, 'r-', lw=2.5, label='Bias²')
ax2.plot(degrees_full, variance, 'b-', lw=2.5, label='Variance')
ax2.plot(degrees_full, total_error_theory, 'k-', lw=3, label='Toplam Hata (Bias²+Var)')
ax2.axvline(degrees_full[np.argmin(total_error_theory)], color='green', ls='--', lw=2,
            label='Optimal Nokta')
ax2.fill_between(degrees_full, 0, total_error_theory, where=(np.array(degrees_full) < 5),
                  alpha=0.2, color='orange', label='Underfitting Bölgesi')
ax2.fill_between(degrees_full, 0, total_error_theory, where=(np.array(degrees_full) > 12),
                  alpha=0.2, color='red', label='Overfitting Bölgesi')
ax2.set_xlabel('Model Karmaşıklığı', fontsize=12)
ax2.set_ylabel('Hata Bileşenleri', fontsize=12)
ax2.set_title('Bias-Variance Trade-off (Teorik)', fontweight='bold')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/bias_variance_tradeoff.png', dpi=120, bbox_inches='tight')
plt.close()

print("═══ BIAS-VARIANCE ANALİZİ ═══")
print(f"Polinom d=1  (Underfitting) → Train MSE={train_errors[0]:.2f}, Test MSE={test_errors[0]:.2f}")
print(f"Polinom d=3  (İyi Fit)      → Train MSE={train_errors[1]:.2f}, Test MSE={test_errors[1]:.2f}")
print(f"Polinom d=15 (Overfitting)  → Train MSE={train_errors[3]:.2f}, Test MSE={test_errors[3]:.2f}")
print(f"\nOptimal derece: {degrees_full[optimal_idx]} (Test MSE minimum)")
print("\nGrafikler: /tmp/bias_variance_polynomial.png, /tmp/bias_variance_tradeoff.png")


## 5.2. Regresyon Analizi: Klasikten Moderne


### B. OLS Çözümü: Normal Equations

`bolum05/05_02_01_b-ols-cozumu-normal-equations.py`

_Kitap: Kod 5.4_


In [ ]:
import random
# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from scipy import stats

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
np.random.seed(42)
X, y = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)
df = pd.DataFrame({'X': X.ravel(), 'Y': y})

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
print("═══ STATSMODELS — İstatistiksel Çıkarım ═══")
X_sm = sm.add_constant(df['X'])
model_sm = sm.OLS(df['Y'], X_sm).fit()
print(model_sm.summary())

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
print("\n═══ SKLEARN — Tahminleme ═══")
X_train, X_test, y_train, y_test = train_test_split(
    df[['X']], df['Y'], test_size=0.2, random_state=42)
model_sk = LinearRegression()
model_sk.fit(X_train, y_train)
y_pred = model_sk.predict(X_test)
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"Test R²: {r2_score(y_test, y_pred):.4f}")

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
# Diagnostics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
residuals = df['Y'] - model_sm.fittedvalues

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
# Residual plot
axes[0,0].scatter(model_sm.fittedvalues, residuals, alpha=0.6)
axes[0,0].axhline(0, color='red', ls='--', lw=2)
axes[0,0].set_title('Residual Plot')

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[0,1])
axes[0,1].set_title('Q-Q Plot')

# --- ▌ Kod Örneği 5.2.1 — OLS: statsmodels vs sklearn ---
plt.tight_layout()
plt.savefig('/tmp/ols_diagnostics.png', dpi=120)
print("\nDiagnostics: /tmp/ols_diagnostics.png")


### A. Gradyan İnişi Matematiği

`bolum05/05_02_02_a-gradyan-inisi-matematigi.py`


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X = np.random.randn(200, 1)
y = 2*X.ravel() + 3 + np.random.randn(200)*0.5

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_b = np.c_[np.ones((len(X_scaled), 1)), X_scaled]

def batch_gd(X, y, lr=0.01, n_iter=1000):
    m, n = X.shape
    theta = np.random.randn(n)
    history = []
    for _ in range(n_iter):
        gradients = (1/m) * X.T.dot(X.dot(theta) - y)
        theta -= lr * gradients
        cost = (1/(2*m)) * np.sum((X.dot(theta) - y)**2)
        history.append(cost)
    return theta, history

def sgd(X, y, lr=0.01, n_epochs=50):
    m, n = X.shape
    theta = np.random.randn(n)
    history = []
    for _ in range(n_epochs):
        for i in range(m):
            gradients = X[i:i+1].T.dot(X[i:i+1].dot(theta) - y[i:i+1])
            theta -= lr * gradients
        cost = (1/(2*m)) * np.sum((X.dot(theta) - y)**2)
        history.append(cost)
    return theta, history

theta_b, hist_b = batch_gd(X_b, y, lr=0.1, n_iter=500)
theta_s, hist_s = sgd(X_b, y, lr=0.01, n_epochs=50)

print(f"Batch GD final θ: {theta_b}")
print(f"SGD final θ: {theta_s}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(hist_b, 'b-', lw=2, label='Batch GD', alpha=0.8)
ax.plot(hist_s, 'r-', lw=2, label='SGD', alpha=0.7)
ax.set_xlabel('Iteration/Epoch')
ax.set_ylabel('Cost (MSE)')
ax.set_title('Gradient Descent Convergence')
ax.legend()
ax.grid(alpha=0.3)
ax.set_yscale('log')
plt.savefig('/tmp/gd_convergence.png', dpi=120)
print("Saved: /tmp/gd_convergence.png")


### C. Elastic Net

`bolum05/05_02_03_c-elastic-net.py`

_Kitap: Kod 5.5_


In [ ]:
import random
# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
np.random.seed(42)
X, y = make_regression(n_samples=100, n_features=50, n_informative=10, noise=20)

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
models = {
    'Ridge(α=1)': Ridge(alpha=1.0),
    'Ridge(α=10)': Ridge(alpha=10.0),
    'Lasso(α=0.1)': Lasso(alpha=0.1),
    'Lasso(α=1)': Lasso(alpha=1.0),
    'ElasticNet': ElasticNet(alpha=0.5, l1_ratio=0.5)
}

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
for name, model in models.items():
    model.fit(X_train_s, y_train)
    score = model.score(X_test_s, y_test)
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-5)
    print(f"{name:15} | R²={score:.3f} | Non-zero={n_nonzero}/50")

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
# Optimal α selection
alphas = np.logspace(-2, 4, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_s, y_train)
print(f"\nRidge optimal α: {ridge_cv.alpha_:.4f}")

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
lasso_cv = LassoCV(alphas=np.logspace(-4, 1, 50), cv=5)
lasso_cv.fit(X_train_s, y_train)
print(f"Lasso optimal α: {lasso_cv.alpha_:.4f}")
print(f"Lasso selected: {np.sum(np.abs(lasso_cv.coef_)>1e-5)}/50 features")

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
# Regularization path
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
coefs_ridge, coefs_lasso = [], []
for alpha in alphas:
    coefs_ridge.append(Ridge(alpha=alpha).fit(X_train_s, y_train).coef_)
    coefs_lasso.append(Lasso(alpha=alpha).fit(X_train_s, y_train).coef_)

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
for i in range(10):
    ax1.plot(alphas, [c[i] for c in coefs_ridge], alpha=0.7)
    ax2.plot(alphas, [c[i] for c in coefs_lasso], alpha=0.7)

# --- ▌ Kod Örneği 5.2.3 — Ridge, Lasso, ElasticNet Karşılaştırma ---
ax1.set_xscale('log'); ax2.set_xscale('log')
ax1.set_title('Ridge Path'); ax2.set_title('Lasso Path (Sparse)')
for ax in [ax1, ax2]:
    ax.set_xlabel('α'); ax.set_ylabel('Coefficient'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/regularization_path.png', dpi=120)
print("Saved: /tmp/regularization_path.png")
